<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:
using System;
using System.Collections.Generic;
using System.Linq;

public delegate void NotifyHandler(string message);
public delegate void ValueChangeHandler(string name, double newValue);

public interface IPerson
{
    string Name { get; }
    string GetInfo();
    void SayHello();
}

public interface IResearcher
{
    void DoResearch();
}

public interface IMentor
{
    void MentorStudent(Student student);
}

public class Person : IPerson
{
    private string _name;
    private int _age;
    private string _gender;

    public string Name
    {
        get => _name;
        set => _name = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Пустое имя");
    }

    public int Age
    {
        get => _age;
        set => _age = (value >= 0 && value <= 120) ? value : throw new ArgumentOutOfRangeException("Неверный возраст");
    }

    public string Gender
    {
        get => _gender;
        set => _gender = (value == "Мужской" || value == "Женский") ? value : throw new ArgumentException("Неверный пол");
    }

    public string Email { get; set; }
    public string Phone { get; set; }
    public string City { get; set; }

    public static List<Person> AllPeople { get; } = new List<Person>();
    public static event NotifyHandler OnPersonCreated;

    public Person(string name, int age, string gender, string email = "нет", string phone = "нет", string city = "Тюмень")
    {
        Name = name;
        Age = age;
        Gender = gender;
        Email = email;
        Phone = phone;
        City = city;

        AllPeople.Add(this);
        OnPersonCreated?.Invoke($"Создан {GetType().Name}: {Name}");
    }

    public virtual string GetInfo() => $"Имя: {Name}, Возраст: {Age}, Пол: {Gender}, Город: {City}";
    public virtual string GetInfo(bool shortForm) => shortForm ? $"{Name} ({Age} лет)" : GetInfo();

    public virtual void SayHello() => Console.WriteLine($"Привет! Меня зовут {Name}.");
    public virtual void SayHello(string lang)
    {
        if (lang == "en") Console.WriteLine($"Hello! My name is {Name}.");
        else SayHello();
    }

    public void UpdateContacts(string email, string phone) { Email = email; Phone = phone; }
    public void MoveToCity(string newCity) => City = newCity;
    public void HaveBirthday() => Age++;
}

public class Student : Person
{
    private string _university;
    private int _course;
    private List<int> _grades = new List<int>();

    public string University
    {
        get => _university;
        set => _university = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Пустой ВУЗ");
    }

    public int Course
    {
        get => _course;
        set => _course = (value >= 1 && value <= 6) ? value : throw new ArgumentOutOfRangeException("Курс 1-6");
    }

    public string Major { get; set; }
    public string Group { get; set; }
    public double Scholarship { get; private set; }

    public event ValueChangeHandler OnGradeAdded;

    public Student(string name, int age, string gender, string university, int course, string major = "ИТ", string group = "ПИ-21")
        : base(name, age, gender)
    {
        University = university;
        Course = course;
        Major = major;
        Group = group;
    }

    public void AddGrade(int grade)
    {
        if (grade >= 2 && grade <= 5)
        {
            _grades.Add(grade);
            Scholarship = GetAverageGrade() >= 4.5 ? 5000 : 0;
            OnGradeAdded?.Invoke(Name, GetAverageGrade());
        }
    }

    public void AddGrade(params int[] grades)
    {
        foreach (var g in grades) AddGrade(g);
    }

    public double GetAverageGrade() => _grades.Count > 0 ? _grades.Average() : 0;
    public void NextCourse() { if (Course < 6) Course++; }
    public void Study() => Console.WriteLine($"{Name} сейчас учится.");

    public override void SayHello() => Console.WriteLine($"Привет! Я студент {Name}, {Course} курс, {University}.");
    public override string GetInfo() => base.GetInfo() + $", ВУЗ: {University}, Курс: {Course}, Группа: {Group}, Ср. балл: {GetAverageGrade():F2}";
}

public class Employee : Person
{
    private string _company;
    private decimal _salary;

    public string Company
    {
        get => _company;
        set => _company = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Пустая компания");
    }

    public decimal Salary
    {
        get => _salary;
        set => _salary = value >= 0 ? value : throw new ArgumentOutOfRangeException("Зарплата < 0");
    }

    public string Position { get; set; }
    public string Department { get; set; }
    public List<string> Projects { get; } = new List<string>();

    public event ValueChangeHandler OnSalaryChanged;

    public Employee(string name, int age, string gender, string company, decimal salary, string position = "Специалист", string department = "Отдел ИТ")
        : base(name, age, gender)
    {
        Company = company;
        Salary = salary;
        Position = position;
        Department = department;
    }

    public void AddProject(string project) => Projects.Add(project);

    public void RaiseSalary(decimal amount)
    {
        Salary += amount;
        OnSalaryChanged?.Invoke(Name, (double)Salary);
    }

    public void RaiseSalary(double percent)
    {
        Salary += Salary * (decimal)(percent / 100.0);
        OnSalaryChanged?.Invoke(Name, (double)Salary);
    }

    public void ChangePosition(string newPos) => Position = newPos;
    public void Work() => Console.WriteLine($"{Name} сейчас работает в {Company}.");

    public override string GetInfo() => base.GetInfo() + $", Компания: {Company}, Должность: {Position}, Зарплата: {Salary} руб.";
}

public class Teacher : Employee
{
    private string _subject;
    private int _experience;
    private List<Student> _students = new List<Student>();

    public string Subject
    {
        get => _subject;
        set => _subject = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Пустой предмет");
    }

    public int Experience
    {
        get => _experience;
        set => _experience = (value >= 0 && value <= 70) ? value : throw new ArgumentOutOfRangeException("Стаж 0-70");
    }

    public string AcademicDegree { get; set; }
    public int Classroom { get; set; }

    public Teacher(string name, int age, string gender, string company, decimal salary, string subject, int experience, string degree = "Кандидат наук", int room = 305)
        : base(name, age, gender, company, salary, "Преподаватель", "Кафедра")
    {
        Subject = subject;
        Experience = experience;
        AcademicDegree = degree;
        Classroom = room;
    }

    public void AddStudent(Student student) => _students.Add(student);
    public void GradeStudent(Student student, int grade)
    {
        if (_students.Contains(student)) student.AddGrade(grade);
    }
    public int GetStudentsCount() => _students.Count;
    public void Teach() => Console.WriteLine($"{Name} преподаёт {Subject}.");

    public override void SayHello() => Console.WriteLine($"Здравствуйте! Я преподаватель {Name}, веду '{Subject}'.");
    public override string GetInfo() => base.GetInfo() + $", Предмет: {Subject}, Стаж: {Experience} лет, Степень: {AcademicDegree}";
}

public class Professor : Teacher, IResearcher, IMentor
{
    public string LabName { get; set; }

    public Professor(string name, int age, string gender, string company, decimal salary, string subject, int experience, string labName)
        : base(name, age, gender, company, salary, subject, experience, "Доктор наук", 401)
    {
        LabName = labName;
    }

    void IResearcher.DoResearch() => Console.WriteLine($"Профессор {Name} проводит исследования в {LabName}.");
    public void MentorStudent(Student student) => Console.WriteLine($"Профессор {Name} курирует студента {student.Name}.");

    public override void SayHello() => Console.WriteLine($"Здравствуйте! Я профессор {Name}, заведующий {LabName}.");
}

public class PersonGroup<T> where T : Person
{
    public string Title { get; set; }
    private List<T> _members = new List<T>();

    public PersonGroup(string title) => Title = title;

    public void Add(T person) => _members.Add(person);
    public double GetAverageAge() => _members.Count > 0 ? _members.Average(p => p.Age) : 0;

    public void ShowAll()
    {
        Console.WriteLine($"Группа '{Title}' (участников: {_members.Count}, ср. возраст: {GetAverageAge():F1}):");
        foreach (var m in _members) Console.WriteLine($"- {m.GetInfo()}");
    }
}

public class PersonCardPrinter
{
    private readonly IPerson _person;

    public PersonCardPrinter(IPerson person)
    {
        _person = person;
    }

    public void Print()
    {
        Console.WriteLine($"Карточка: {_person.GetInfo()}");
        _person.SayHello();
    }
}

Person.OnPersonCreated += msg => Console.WriteLine($"[Событие] {msg}");

var student1 = new Student("Анна Смирнова", 20, "Женский", "ТюмГУ", 3);
var student2 = new Student("Олег Новиков", 22, "Мужской", "ТюмГУ", 4);
var employee = new Employee("Пётр Сидоров", 35, "Мужской", "Яндекс", 120000, "Разработчик");
var teacher = new Teacher("Елена Иванова", 45, "Женский", "ТюмГУ", 85000, "Математика", 15);
var prof = new Professor("Сергей Николаев", 52, "Мужской", "ТюмГУ", 150000, "ИИ", 25, "Лаборатория ИИ");

student1.OnGradeAdded += (name, avg) => Console.WriteLine($"[Событие] У студента {name} новый средний балл: {avg:F2}");
employee.OnSalaryChanged += (name, sal) => Console.WriteLine($"[Событие] Новая зарплата {name}: {sal} руб.");

Console.WriteLine("\nОценки и зарплата:");
student1.AddGrade(5, 4, 5);
student2.AddGrade(4);
employee.RaiseSalary(15000m);

Console.WriteLine("\nРабота с коллекциями (LINQ):");
var goodStudents = Person.AllPeople.OfType<Student>().Where(s => s.GetAverageGrade() >= 4.5);
Console.WriteLine("Отличники и хорошисты (балл >= 4.5):");
foreach (var s in goodStudents)
{
Console.WriteLine($"- {s.Name} (ср. балл: {s.GetAverageGrade():F2}, стипендия: {s.Scholarship} руб.)");
}
double totalScholarship = Person.AllPeople.OfType<Student>().Sum(s => s.Scholarship);
Console.WriteLine($"Общий фонд стипендий: {totalScholarship} руб.");
Console.WriteLine($"Всего людей в базе: {Person.AllPeople.Count}");

[Событие] Создан Student: Анна Смирнова
[Событие] Создан Student: Олег Новиков
[Событие] Создан Employee: Пётр Сидоров
[Событие] Создан Teacher: Елена Иванова
[Событие] Создан Professor: Сергей Николаев

Оценки и зарплата:
[Событие] У студента Анна Смирнова новый средний балл: 5.00
[Событие] У студента Анна Смирнова новый средний балл: 4.50
[Событие] У студента Анна Смирнова новый средний балл: 4.67
[Событие] Новая зарплата Пётр Сидоров: 135000 руб.

Работа с коллекциями (LINQ):
Отличники и хорошисты (балл >= 4.5):
- Анна Смирнова (ср. балл: 4.67, стипендия: 5000 руб.)
Общий фонд стипендий: 5000 руб.
Всего людей в базе: 5
